# Efficient Edge AI - MNIST hands-on with PyTorch

**Goal:** compare a dense Multi-Layer Perceptron (MLP) with a small CNN on **MNIST** handwritten digits. Both models classify grayscale $28\times28$ images into the digits 0-9.

You will implement and train both models, then compare:

- training/validation curves;
- test accuracy;
- number of parameters;
- parameter memory and serialized state-dict size;
- MACs per input image;
- batch-1 inference latency.

## Architectures

**Dense MLP**

`1×28×28 -> Flatten -> Linear(128) -> Linear(64) -> Linear(10)`

**Small CNN**

`1×28×28 -> Conv3×3(8) -> MaxPool2 -> Conv3×3(16) -> MaxPool2 -> Flatten -> Linear(32) -> Linear(10)`

Use ReLU after every hidden Conv/Linear layer. The final layer outputs **logits**, not probabilities. We keep this architecture intentionally small so its cost is suitable for an edge-AI comparison.

Suggested time: **45-60 min**.

## 0. Running in Google Colab

[Open Google Colab](https://colab.research.google.com/), then choose **File -> Upload notebook** and select this `.ipynb` file. Alternatively, after publishing the repository on GitHub, replace the button URL with `https://colab.research.google.com/github/alessandrocapotondi/edge_ai_course/blob/main/edge_ai_mnist_pytorch_hands_on.ipynb`.

For the latency comparison, select **Runtime -> Change runtime type -> GPU** in Colab. MNIST also works on a CPU; a GPU only makes training and measurements faster.

We do not use data augmentation: `ToTensor()` converts pixels from `[0, 255]` to float tensors in `[0, 1]`.

In [ ]:
import os
import time
import random
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3
NUM_CLASSES = 10

## 1. Load MNIST

`torchvision.datasets.MNIST` downloads 60,000 grayscale handwritten digits, each with shape `1×28×28`. We reserve 55,000 examples for training and 5,000 for validation, and use the official 10,000-image test set.

In [ ]:
# ToTensor() returns float images with shape (channels, height, width).
transform = transforms.ToTensor()

# MNIST is grayscale: each image has one channel and measures 28 x 28.
full_train = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_set = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# Keep a separate, reproducible validation set.
generator = torch.Generator().manual_seed(SEED)
train_set, val_set = random_split(
    full_train, [55_000, 5_000], generator=generator
)

pin_memory = torch.cuda.is_available()
train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=pin_memory
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=pin_memory
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=pin_memory
)

print("Train:", len(train_set))
print("Val:  ", len(val_set))
print("Test: ", len(test_set))

In [ ]:
class_names = [str(digit) for digit in range(NUM_CLASSES)]
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), images[:10], labels[:10]):
    # image[0] removes the single channel before displaying the grayscale image.
    ax.imshow(image[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(class_names[int(label)])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Exercise - implement the MLP

Build exactly `Flatten -> Linear(784, 128) -> ReLU -> Linear(128, 64) -> ReLU -> Linear(64, 10)`.

1. In `__init__`, assign an `nn.Sequential` containing the seven layers in the specified order to `self.net`.
2. `nn.Flatten()` transforms each `N×1×28×28` input into `N×784`, keeping `N` as the batch dimension.
3. In `forward`, return `self.net(x)` without adding `softmax`: `CrossEntropyLoss` receives logits.

Do not add dropout or batch normalization: they would alter the controlled comparison.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Replace the TODO and exception with this sequential stack:
        # self.net = nn.Sequential(
        #     nn.Flatten(),                 # N x 1 x 28 x 28 -> N x 784
        #     nn.Linear(28 * 28, 128),      # first hidden representation
        #     nn.ReLU(),                    # non-linearity after hidden layer
        #     nn.Linear(128, 64),
        #     nn.ReLU(),
        #     nn.Linear(64, NUM_CLASSES),   # 10 raw logits, no softmax
        # )
        raise NotImplementedError("Implement MLP.__init__()")

    def forward(self, x):
        # Return the logits produced by the layer stack:
        # return self.net(x)
        raise NotImplementedError("Implement MLP.forward()")

# mlp = MLP().to(DEVICE)
# print(mlp)

### Self-check for the MLP

In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())

@torch.no_grad()
def check_mlp(model):
    model.eval()
    # MNIST batches contain one 28 x 28 grayscale channel.
    logits = model(torch.zeros(1, 1, 28, 28, device=DEVICE))
    assert tuple(logits.shape) == (1, NUM_CLASSES), logits.shape
    assert count_parameters(model) == 109_386, (
        f"Unexpected parameter count: {count_parameters(model):,}"
    )
    print("MLP architecture looks correct.")
    print(f"Parameters: {count_parameters(model):,}")

# check_mlp(mlp)

## 3. Exercise - implement the small CNN

Implement `Conv3×3(1->8, padding=1) -> ReLU -> MaxPool2×2 -> Conv3×3(8->16, padding=1) -> ReLU -> MaxPool2×2 -> Flatten -> Linear(784->32) -> ReLU -> Linear(32->10)`.

Follow these steps:

1. Create `self.features` with the two convolutions, ReLU activations, and pooling layers.
2. Use `padding=1` to preserve spatial dimensions in the convolutions.
3. The two MaxPool layers halve the spatial dimensions $28	o14	o7$: after `self.features`, the shape is `N×16×7×7`.
4. Create `self.classifier` with `Flatten`, the `16*7*7 -> 32` linear layer, ReLU, and the 10-logit output.
5. In `forward`, pass the input through `self.features` first and then through `self.classifier`.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Replace the TODO and exception with the feature extractor:
        # self.features = nn.Sequential(
        #     nn.Conv2d(1, 8, kernel_size=3, stride=1, padding=1),  # 28 x 28
        #     nn.ReLU(),
        #     nn.MaxPool2d(kernel_size=2, stride=2),                 # 14 x 14
        #     nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1),
        #     nn.ReLU(),
        #     nn.MaxPool2d(kernel_size=2, stride=2),                 # 7 x 7
        # )
        # Then create self.classifier with Flatten, Linear(16 * 7 * 7, 32),
        # ReLU, and Linear(32, NUM_CLASSES).
        raise NotImplementedError("Implement SmallCNN.__init__()")

    def forward(self, x):
        # First extract convolutional features, then return classifier logits.
        # return self.classifier(self.features(x))
        raise NotImplementedError("Implement SmallCNN.forward()")

# cnn = SmallCNN().to(DEVICE)
# print(cnn)

### Self-check for the CNN

In [ ]:
@torch.no_grad()
def check_cnn(model):
    model.eval()
    x = torch.zeros(1, 1, 28, 28, device=DEVICE)
    logits = model(x)
    assert tuple(logits.shape) == (1, NUM_CLASSES), logits.shape
    assert count_parameters(model) == 26_698, (
        f"Unexpected parameter count: {count_parameters(model):,}"
    )
    print("CNN architecture looks correct.")
    print(f"Parameters: {count_parameters(model):,}")

# check_cnn(cnn)

## 4. Training utilities

We use the same setup for both models:

- Adam, learning rate `1e-3`;
- `CrossEntropyLoss` on logits;
- same batch size and number of epochs.

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = loss_fn(logits, y)

        if is_training:
            loss.backward()
            optimizer.step()

        batch_size = x.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_examples += batch_size

    return total_loss / total_examples, total_correct / total_examples


def fit(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {
        "loss": [], "accuracy": [],
        "val_loss": [], "val_accuracy": []
    }

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(
            model, train_loader, loss_fn, optimizer
        )
        with torch.no_grad():
            val_loss, val_acc = run_epoch(
                model, val_loader, loss_fn, optimizer=None
            )

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f}"
        )

    return history

## 5. Exercise — train both models

Instantiate, self-check, and train both architectures.

In [ ]:
# TODO:
# 1) instantiate mlp and cnn on DEVICE
# 2) run check_mlp() and check_cnn()
# 3) train both models using fit()
#
# Expected variables at the end:
#   mlp, cnn, history_mlp, history_cnn
#
# Skeleton:
#
# mlp = MLP().to(DEVICE)
# cnn = SmallCNN().to(DEVICE)
# check_mlp(mlp)
# check_cnn(cnn)
#
# history_mlp = fit(mlp, train_loader, val_loader)
# history_cnn = fit(cnn, train_loader, val_loader)

## 6. Compare training curves

Look for convergence speed, final validation accuracy, train/validation gap, and possible overfitting.

In [ ]:
def plot_histories(history_mlp, history_cnn):
    epochs_mlp = range(1, len(history_mlp["loss"]) + 1)
    epochs_cnn = range(1, len(history_cnn["loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_mlp, history_mlp["loss"], label="MLP train")
    plt.plot(epochs_mlp, history_mlp["val_loss"], label="MLP val")
    plt.plot(epochs_cnn, history_cnn["loss"], label="CNN train")
    plt.plot(epochs_cnn, history_cnn["val_loss"], label="CNN val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training curves — loss")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs_mlp, history_mlp["accuracy"], label="MLP train")
    plt.plot(epochs_mlp, history_mlp["val_accuracy"], label="MLP val")
    plt.plot(epochs_cnn, history_cnn["accuracy"], label="CNN train")
    plt.plot(epochs_cnn, history_cnn["val_accuracy"], label="CNN val")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training curves — accuracy")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()

# plot_histories(history_mlp, history_cnn)

## 7. Test accuracy

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    loss_fn = nn.CrossEntropyLoss()
    return run_epoch(model, loader, loss_fn, optimizer=None)

# mlp_test_loss, mlp_test_acc = evaluate(mlp, test_loader)
# cnn_test_loss, cnn_test_acc = evaluate(cnn, test_loader)
# print(f"MLP test accuracy: {mlp_test_acc:.4f}")
# print(f"CNN test accuracy: {cnn_test_acc:.4f}")

## 8. Model size in memory and on disk

We report:

1. **Parameter footprint** = bytes occupied by parameters according to their dtype.
2. **Serialized state-dict size** = actual `.pt` file size.

These values do **not** include activations, temporary workspaces, CUDA allocator overhead, or optimizer state.

In [ ]:
def parameter_memory_bytes(model):
    return sum(p.numel() * p.element_size() for p in model.parameters())

def serialized_state_dict_size_bytes(model, filename):
    path = Path(tempfile.gettempdir()) / filename
    torch.save(model.state_dict(), path)
    return path.stat().st_size

# mlp_param_bytes = parameter_memory_bytes(mlp)
# cnn_param_bytes = parameter_memory_bytes(cnn)
# mlp_file_bytes = serialized_state_dict_size_bytes(mlp, "mlp_state.pt")
# cnn_file_bytes = serialized_state_dict_size_bytes(cnn, "cnn_state.pt")
#
# print("MLP parameter footprint [MiB]:", mlp_param_bytes / 2**20)
# print("CNN parameter footprint [MiB]:", cnn_param_bytes / 2**20)

## 9. Count MACs per image with forward hooks

We count MACs for `nn.Conv2d` and `nn.Linear`:

- Conv2d: `Hout × Wout × Cout × Kh × Kw × (Cin/groups)`
- Linear: `Nin × Nout`

Bias additions, ReLU, pooling, and data movement are excluded.

In [ ]:
@torch.no_grad()
def count_macs_pytorch(model, input_shape=(1, 1, 28, 28), verbose=True):
    model.eval()
    rows = []
    hooks = []

    def hook_fn(name):
        def hook(module, inputs, output):
            macs = 0
            if isinstance(module, nn.Conv2d):
                # Output shape includes batch; MACs are reported for one image.
                _, c_out, h_out, w_out = output.shape
                k_h, k_w = module.kernel_size
                c_in_per_group = module.in_channels // module.groups
                macs = int(h_out * w_out * c_out * k_h * k_w * c_in_per_group)
            elif isinstance(module, nn.Linear):
                macs = int(module.in_features * module.out_features)

            rows.append({
                "layer": name,
                "type": module.__class__.__name__,
                "output_shape": tuple(output.shape),
                "MACs": macs,
            })
        return hook

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            hooks.append(module.register_forward_hook(hook_fn(name)))

    dummy = torch.zeros(*input_shape, device=DEVICE)
    _ = model(dummy)

    for hook in hooks:
        hook.remove()

    total_macs = sum(row["MACs"] for row in rows)
    if verbose:
        display(pd.DataFrame(rows))
        print(f"Total MACs / image: {total_macs:,}")
    return total_macs

# mlp_macs = count_macs_pytorch(mlp)
# cnn_macs = count_macs_pytorch(cnn)

### Manual MAC check

First calculate the MACs by hand and compare them with `count_macs_pytorch()`. One MAC is a multiplication plus an accumulation; bias, ReLU, and pooling are excluded.

**CNN**

- Conv1: `28 x 28 x 8 x 3 x 3 x 1 = ?`
- Conv2: `14 x 14 x 16 x 3 x 3 x 8 = ?`
- Linear32: `784 x 32 = ?`
- Linear10: `32 x 10 = ?`

**MLP**

- Linear128: `784 x 128 = ?`
- Linear64: `128 x 64 = ?`
- Linear10: `64 x 10 = ?`

## 10. Batch-1 inference latency

We benchmark one image after warm-up. On CUDA, explicit synchronization is required because kernel launches are asynchronous.

Compare MLP vs CNN on the **same runtime**. Cross-framework latency differences also reflect framework/compiler/runtime effects.

In [ ]:
@torch.no_grad()
def benchmark_latency_ms(model, x_sample, warmup=30, runs=200):
    model.eval()
    x_sample = x_sample.to(DEVICE)

    for _ in range(warmup):
        _ = model(x_sample)

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    times_ms = []
    for _ in range(runs):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        _ = model(x_sample)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        times_ms.append((t1 - t0) * 1e3)

    times_ms = np.asarray(times_ms)
    return {
        "mean_ms": float(times_ms.mean()),
        "median_ms": float(np.median(times_ms)),
        "p95_ms": float(np.percentile(times_ms, 95)),
    }

# x_one, _ = test_set[0]
# x_one = x_one.unsqueeze(0)
# mlp_latency = benchmark_latency_ms(mlp, x_one)
# cnn_latency = benchmark_latency_ms(cnn, x_one)
# print("MLP:", mlp_latency)
# print("CNN:", cnn_latency)

## 11. Final comparison table

In [ ]:
def mib(n_bytes):
    return n_bytes / (2**20)

# TODO: uncomment after training + measurements.
#
# results = pd.DataFrame([
#     {
#         "model": "MLP",
#         "test_accuracy": mlp_test_acc,
#         "parameters": count_parameters(mlp),
#         "parameter_MiB": mib(mlp_param_bytes),
#         "state_dict_MiB": mib(mlp_file_bytes),
#         "MACs_per_image": mlp_macs,
#         "latency_median_ms": mlp_latency["median_ms"],
#         "latency_p95_ms": mlp_latency["p95_ms"],
#     },
#     {
#         "model": "CNN",
#         "test_accuracy": cnn_test_acc,
#         "parameters": count_parameters(cnn),
#         "parameter_MiB": mib(cnn_param_bytes),
#         "state_dict_MiB": mib(cnn_file_bytes),
#         "MACs_per_image": cnn_macs,
#         "latency_median_ms": cnn_latency["median_ms"],
#         "latency_p95_ms": cnn_latency["p95_ms"],
#     },
# ]).set_index("model")
#
# display(results.style.format({
#     "test_accuracy": "{:.4f}",
#     "parameter_MiB": "{:.3f}",
#     "state_dict_MiB": "{:.3f}",
#     "MACs_per_image": "{:,.0f}",
#     "latency_median_ms": "{:.3f}",
#     "latency_p95_ms": "{:.3f}",
# }))

## 12. Discussion questions

Base the answers on **your measured results**.

1. The MLP and CNN have similar MAC counts. Why can their test accuracy be very different?
2. Which model has more parameters, and where are they concentrated?
3. Does fewer parameters automatically mean lower latency?
4. Why can two models with similar MAC counts have different GPU latency?
5. Why is the `.pt` file size not exactly `4 × #parameters`?
6. If all weights were converted from FP32 to INT8, what would happen to the weight-memory footprint in the ideal case?
7. Which of the measured metrics are architecture-level properties, and which strongly depend on hardware/runtime?

## 13. Optional extension

Benchmark batch sizes `1`, `8`, `32`, and `128`, then report:

- batch latency;
- milliseconds per image;
- throughput in images/s.

This highlights the distinction between **latency** and **throughput**, which becomes crucial in edge deployment.

In [ ]:
# OPTIONAL EXERCISE
#
# Implement a multi-batch latency benchmark and return a DataFrame with:
# batch_size, batch_latency_ms, ms_per_image, images_per_second

---
### Checks and references

These values do not depend on training:

- MLP: **109,386 parameters**, **109,184 MAC/image**;
- CNN: **26,698 parameters**, **307,648 MAC/image**.

Accuracy and latency are empirical: report the values obtained from your run.

### Official documentation

- [PyTorch: building models with nn.Module](https://pytorch.org/docs/stable/notes/modules.html)
- [torchvision.datasets.MNIST](https://pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html)
- [PyTorch CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [Google Colab guide](https://colab.research.google.com/notebooks/intro.ipynb)

## 14. Optional task - extend to CIFAR-10

After completing MNIST, try CIFAR-10. The images are RGB `3×32×32`: update both the data loading and the model dimensions. Keep training, metrics, and benchmarks identical for a fair comparison. The following cell provides a starting point without changing the main MNIST workflow.

In [ ]:
# OPTIONAL: CIFAR-10 extension
# 1) Load RGB data. This replaces MNIST only inside this optional experiment.
# cifar_transform = transforms.ToTensor()
# cifar_train = datasets.CIFAR10("./data", train=True, download=True, transform=cifar_transform)
# cifar_test = datasets.CIFAR10("./data", train=False, download=True, transform=cifar_transform)
#
# 2) Copy the two MNIST classes and adapt the layers below:
# # CifarMLP: nn.Linear(32 * 32 * 3, 128) is the first dense layer.
# # CifarCNN: first convolution becomes nn.Conv2d(3, 8, 3, padding=1).
# # After two pools: 32 -> 16 -> 8, so use nn.Linear(16 * 8 * 8, 32).
#
# 3) Split cifar_train into 45,000 training and 5,000 validation examples,
# #    create DataLoaders, and reuse fit(), evaluate(), count_macs_pytorch(),
# #    and benchmark_latency_ms() to compare CIFAR-10 against the MNIST results.